In [1]:
# =====================================================================
# OmniVoice TTS — self-contained WebSocket test (edit params & re-run)
# =====================================================================
# %pip install -q websockets   # uncomment if websockets isn't in this kernel

import asyncio, json, time, uuid, wave
import websockets
from IPython.display import Audio, display

In [13]:

# ---- params ----
HOST, PORT, PATH_PREFIX = "172.16.1.4", 80, "/omnivoice-tts"   # via nginx
# direct (no nginx): HOST, PORT, PATH_PREFIX = "127.0.0.1", 8080, ""
VOICE   = "saavi-assamese"
LANG    = "as"        # match the text's language: as=Assamese, bn=Bengali, hi=Hindi, en=English
SPEED   = None        # e.g. 1.1
OUT     = "saavi_assamese.wav"
TEXT    = "নমস্কাৰ ছাৰ, মই বাজাজ ফাইনান্স বেংকৰ পৰা অংকিতা কৈ আছোঁ। আপোনাৰ ২০০০ টকাৰ ঋণৰ E M I এতিয়াও বাকি আছে, যিটোৰ শেষ তাৰিখ ৫ জুলাই, ২০২৬। ছাৰ, আপুনি কেতিয়া এই টকাখিনি পৰিশোধ কৰিব?"
# TEXT    = "hello, i'm aarav from big basket. rohil asked brahmanandam reddy to inform avinash that swastik, pranav, and chandrasekhar should come over, so akshay, vignesh, and adithyan can address the issue with kirtiman, raghav, and krishnakumar."

In [ ]:

def _split(raw):
    """Split a combined binary frame ({json header} + raw PCM) -> (msg, pcm_bytes)."""
    if isinstance(raw, str):
        return json.loads(raw), b""
    depth = end = 0
    for i, b in enumerate(raw):
        if b == 0x7B: depth += 1          # '{'
        elif b == 0x7D:                   # '}'
            depth -= 1
            if depth == 0:
                end = i + 1; break
    return json.loads(raw[:end]), raw[end:]

async def synth():
    call_id = f"nb-{uuid.uuid4().hex[:8]}"
    url = f"ws://{HOST}:{PORT}{PATH_PREFIX.rstrip('/')}/ws/{call_id}"
    print("connecting", url)
    pcm, sr, ttfb, saw_final, done = bytearray(), 24000, None, False, {}
    req = {"type": "synthesize", "call_id": call_id, "text_id": uuid.uuid4().hex[:8],
           "text": TEXT, "streaming": True}
    if VOICE: req["voice_id"] = VOICE
    if LANG:  req["language"] = LANG
    if SPEED: req["speed"] = SPEED

    async with websockets.connect(url, max_size=100*1024*1024, open_timeout=10,
                                  close_timeout=3, ping_interval=None) as ws:
        t0 = time.perf_counter()
        await ws.send(json.dumps(req))
        while True:
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=(5 if saw_final else 120))
            except asyncio.TimeoutError:
                if saw_final: break
                print("TIMEOUT waiting for audio"); return None
            msg, audio = _split(raw)
            mt = msg.get("type")
            if mt == "audio_chunk":
                if not audio:
                    audio = await ws.recv()
                    if isinstance(audio, str): audio = audio.encode()
                if ttfb is None:
                    ttfb = time.perf_counter() - t0
                    print(f"first chunk in {ttfb*1000:.0f} ms  cache_hit={msg.get('cache_hit')}")
                sr = msg.get("sample_rate", sr); pcm += audio
                if msg.get("is_final"): saw_final = True
            elif mt == "audio_done":
                done = msg; break
            elif mt == "error":
                print("SERVER ERROR:", msg.get("error")); return None

    total = time.perf_counter() - t0
    audio_s = (len(pcm)//2)/sr if sr else 0
    print(f"done  chunks={done.get('chunks')}  audio={audio_s:.2f}s  "
          f"ttfb={(ttfb*1000 if ttfb else 0):.0f}ms  total={total*1000:.0f}ms  "
          f"rtf={done.get('rtf')}  bytes={len(pcm)}  sr={sr}")
    with wave.open(OUT, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(bytes(pcm))
    print("wrote", OUT)
    return OUT

In [14]:

out = await synth()          # Jupyter supports top-level await
if out: display(Audio(filename=out))

connecting ws://172.16.1.4:80/omnivoice-tts/ws/nb-d080847d
first chunk in 735 ms  cache_hit=False
done  chunks=2  audio=15.03s  ttfb=735ms  total=759ms  rtf=0.048  bytes=721440  sr=24000
wrote saavi_assamese.wav
